In [1]:
# Block 1: load raw data files
import pandas as pd
base = 'C:/Users/cigde/Dropbox/Habituation_paper/October_Emily/'
inp, lab = base + 'input_arrays/pls_regression/', base + 'final_figures/'
erp  = pd.read_csv(inp + 'cluster_RS_all_ages.csv',   dtype={'UniqueID': str})
comp = pd.read_csv(inp + 'component_RS_all_ages.csv', dtype={'UniqueID': str})
spec = pd.read_csv(inp + 'spectral_RS_all_ages.csv',  dtype={'UniqueID': str})

In [2]:
# Block 2: build temporal feature table
temporal = erp.merge(comp, on='UniqueID')
cols = [c for c in temporal.columns if c.startswith('frontocentral_') or (c.endswith('_diff') and not c.startswith('N2'))]
temporal = temporal[['UniqueID'] + cols]

In [4]:
# Block 3: build power and phase feature tables
power = spec[['UniqueID'] + [c for c in spec.columns if c.startswith('power_') and c.endswith('poststim_diff')]]
phase = spec[['UniqueID'] + [c for c in spec.columns if c.startswith('itpc_') and c.endswith('poststim_diff')]]

In [5]:
# Block 4: define summary function and run temporal k-means clusters
# Block 4: define summary function and run temporal k-means clusters
def summary(label_file, df):
    m = pd.read_csv(lab + label_file, dtype={'UniqueID': str}).merge(df, on='UniqueID').drop(columns='UniqueID')
    print(m.groupby('cluster').agg(['mean', 'std']).T.round(3))

summary('labels_temporal_kmeans.csv', temporal)

cluster                           0       1
frontocentral_W50_100  mean  -0.735   0.067
                       std    0.524   0.578
frontocentral_W100_150 mean  -0.795   0.223
                       std    0.528   0.585
frontocentral_W150_200 mean  -0.818   0.293
                       std    0.705   0.761
frontocentral_W200_250 mean  -1.214  -0.052
                       std    0.724   0.805
frontocentral_W250_300 mean  -1.323  -0.224
                       std    0.890   0.894
P1_amp_diff            mean  -1.150   0.169
                       std    1.069   0.996
N1_amp_diff            mean  -1.435   0.221
                       std    1.063   1.162
P2_amp_diff            mean  -1.215   0.541
                       std    1.253   1.168
P1_lat_diff            mean   0.041  -0.351
                       std    8.718   9.731
N1_lat_diff            mean   2.385  -1.470
                       std   10.239  11.656
P2_lat_diff            mean   3.757   2.222
                       std   13.

In [6]:
# Block 5: power k-means clusters
summary('labels_spectral_kmeans_power_only.csv', power)

cluster                                  0      1
power_slow_poststim_diff       mean  0.058  0.441
                               std   0.646  0.674
power_fast_theta_poststim_diff mean -0.179  0.818
                               std   0.664  0.721
power_alpha_poststim_diff      mean -0.268  0.731
                               std   0.634  0.650
power_low_beta_poststim_diff   mean -0.313  0.352
                               std   0.487  0.547
power_high_beta_poststim_diff  mean -0.186  0.165
                               std   0.487  0.546


In [7]:
# Block 6: phase k-means clusters
summary('labels_spectral_kmeans_phase_only.csv', phase)

cluster                                 0      1
itpc_slow_poststim_diff       mean -0.007  0.021
                              std   0.059  0.053
itpc_fast_theta_poststim_diff mean -0.029  0.035
                              std   0.040  0.043
itpc_alpha_poststim_diff      mean -0.024  0.037
                              std   0.040  0.036
itpc_low_beta_poststim_diff   mean -0.012  0.018
                              std   0.038  0.035
itpc_high_beta_poststim_diff  mean -0.002  0.002
                              std   0.031  0.030


In [8]:
# Block 7: raw means table for all feature sets
import os
save_path = 'C:/Users/cigde/Dropbox/Habituation_paper/October_Emily/final_figures/figure_6/'
os.makedirs(save_path, exist_ok=True)

rows = []
for f, df in [('labels_temporal_kmeans.csv', temporal),
              ('labels_spectral_kmeans_power_only.csv', power),
              ('labels_spectral_kmeans_phase_only.csv', phase)]:
    m = pd.read_csv(lab + f, dtype={'UniqueID': str}).merge(df, on='UniqueID').drop(columns='UniqueID')
    g = m.groupby('cluster').agg(['mean', 'std'])
    for feat in m.columns.drop('cluster'):
        rows.append({'Feature': feat,
                     'Cluster 1': f"{g.loc[0, (feat, 'mean')]:.3f} ± {g.loc[0, (feat, 'std')]:.3f}",
                     'Cluster 2': f"{g.loc[1, (feat, 'mean')]:.3f} ± {g.loc[1, (feat, 'std')]:.3f}"})
table = pd.DataFrame(rows)
table.to_csv(save_path + 'raw_means_kmeans_clusters.csv', index=False)
print(table.to_string(index=False))

                       Feature      Cluster 1       Cluster 2
         frontocentral_W50_100 -0.735 ± 0.524   0.067 ± 0.578
        frontocentral_W100_150 -0.795 ± 0.528   0.223 ± 0.585
        frontocentral_W150_200 -0.818 ± 0.705   0.293 ± 0.761
        frontocentral_W200_250 -1.214 ± 0.724  -0.052 ± 0.805
        frontocentral_W250_300 -1.323 ± 0.890  -0.224 ± 0.894
                   P1_amp_diff -1.150 ± 1.069   0.169 ± 0.996
                   N1_amp_diff -1.435 ± 1.063   0.221 ± 1.162
                   P2_amp_diff -1.215 ± 1.253   0.541 ± 1.168
                   P1_lat_diff  0.041 ± 8.718  -0.351 ± 9.731
                   N1_lat_diff 2.385 ± 10.239 -1.470 ± 11.656
                   P2_lat_diff 3.757 ± 13.217  2.222 ± 13.963
      power_slow_poststim_diff  0.058 ± 0.646   0.441 ± 0.674
power_fast_theta_poststim_diff -0.179 ± 0.664   0.818 ± 0.721
     power_alpha_poststim_diff -0.268 ± 0.634   0.731 ± 0.650
  power_low_beta_poststim_diff -0.313 ± 0.487   0.352 ± 0.547
 power_h